In [1]:
# ============================================================
# ETL - F_pedidos.csv
# Ajustado al archivo real
# ============================================================

import pandas as pd
import re

# ------------------------------------------------------------
# 1. URL
# ------------------------------------------------------------
url_pedidos = "https://raw.githubusercontent.com/kmeza3278/etl-data-pipeline-17-3278-2021-/refs/heads/main/data/F_pedidos.csv"

# ------------------------------------------------------------
# 2. CARGA
# ------------------------------------------------------------
pedidos_raw = pd.read_csv(url_pedidos)

print("Archivo cargado")
print("Dimensiones:", pedidos_raw.shape)
display(pedidos_raw.head())

# ------------------------------------------------------------
# 3. COPIA
# ------------------------------------------------------------
pedidos = pedidos_raw.copy()

# ------------------------------------------------------------
# 4. FUNCIONES BASE
# ------------------------------------------------------------
def normalizar_columnas(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df

def limpiar_texto(df):
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace(["nan", "None", "NULL", "null", ""], pd.NA)
    return df

def convertir_fecha(col):
    return pd.to_datetime(col, errors="coerce")

def limpiar_monto(valor):
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip()
    valor = valor.replace(",", "")

    try:
        return float(valor)
    except:
        return pd.NA

def normalizar_estado(valor):
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip().lower()

    mapa = {
        "pendiente": "Pendiente",
        "enviado": "Enviado",
        "cancelado": "Cancelado",
        "entregado": "Entregado",
        "procesando": "Procesando"
    }

    return mapa.get(valor, valor.title())

def id_pedido_valido(valor):
    if pd.isna(valor):
        return False
    return bool(re.fullmatch(r"PED\d+", str(valor).strip().upper()))

def id_cliente_valido(valor):
    if pd.isna(valor):
        return False
    return bool(re.fullmatch(r"CL\d+", str(valor).strip().upper()))

def monto_valido(valor):
    if pd.isna(valor):
        return False
    try:
        return float(valor) > 0
    except:
        return False

def motivo_rechazo(row):
    motivos = []

    if pd.isna(row["id_pedido"]):
        motivos.append("id_pedido_vacio")
    elif not id_pedido_valido(row["id_pedido"]):
        motivos.append("id_pedido_invalido")

    if pd.isna(row["id_cliente"]):
        motivos.append("id_cliente_vacio")
    elif not id_cliente_valido(row["id_cliente"]):
        motivos.append("id_cliente_invalido")

    if pd.isna(row["fecha_pedido"]):
        motivos.append("fecha_pedido_invalida")

    if pd.isna(row["monto"]):
        motivos.append("monto_vacio")
    elif not monto_valido(row["monto"]):
        motivos.append("monto_invalido")

    if pd.isna(row["estado"]):
        motivos.append("estado_vacio")

    return ",".join(motivos)

# ------------------------------------------------------------
# 5. LIMPIEZA BASE
# ------------------------------------------------------------
pedidos = normalizar_columnas(pedidos)
pedidos = limpiar_texto(pedidos)
pedidos = pedidos.drop_duplicates()

print("\nColumnas reales:", list(pedidos.columns))

# ------------------------------------------------------------
# 6. LIMPIEZA ESPECIFICA
# ------------------------------------------------------------
pedidos["id_pedido"] = pedidos["id_pedido"].astype("string").str.strip().str.upper()
pedidos["id_cliente"] = pedidos["id_cliente"].astype("string").str.strip().str.upper()
pedidos["fecha_pedido"] = convertir_fecha(pedidos["fecha_pedido"])
pedidos["monto"] = pedidos["monto"].apply(limpiar_monto)
pedidos["estado"] = pedidos["estado"].apply(normalizar_estado)

for col in ["id_pedido", "id_cliente", "estado"]:
    pedidos[col] = pedidos[col].replace(["", "nan", "None", "<NA>"], pd.NA)

# ------------------------------------------------------------
# 7. VALIDACION
# ------------------------------------------------------------
condicion_valida = (
    pedidos["id_pedido"].notna() &
    pedidos["id_cliente"].notna() &
    pedidos["fecha_pedido"].notna() &
    pedidos["monto"].notna() &
    pedidos["estado"].notna() &
    pedidos["id_pedido"].apply(id_pedido_valido) &
    pedidos["id_cliente"].apply(id_cliente_valido) &
    pedidos["monto"].apply(monto_valido)
)

pedidos_curated = pedidos[condicion_valida].copy()
pedidos_rejects = pedidos[~condicion_valida].copy()

pedidos_rejects["motivo_rechazo"] = pedidos_rejects.apply(motivo_rechazo, axis=1)

# ------------------------------------------------------------
# 8. RESULTADOS
# ------------------------------------------------------------
print("\n=== RESUMEN ===")
print("Total procesados:", len(pedidos))
print("Curated:", len(pedidos_curated))
print("Rejects:", len(pedidos_rejects))

print("\n=== ESTADOS NORMALIZADOS ===")
display(pedidos["estado"].value_counts(dropna=False))

print("\n=== MUESTRA CURATED ===")
display(pedidos_curated.head())

print("\n=== MUESTRA REJECTS ===")
display(pedidos_rejects.head())

# ------------------------------------------------------------
# 9. EXPORTAR
# ------------------------------------------------------------
pedidos_curated.to_csv("pedidos_curated.csv", index=False)
pedidos_rejects.to_csv("pedidos_rejects.csv", index=False)

print("\nArchivos generados:")
print("- pedidos_curated.csv")
print("- pedidos_rejects.csv")

from google.colab import files

files.download("pedidos_curated.csv")
files.download("pedidos_rejects.csv")

Archivo cargado
Dimensiones: (236, 5)


,id_pedido,id_cliente,fecha_pedido,monto,estado
0,PED7000,CL1138,2024-11-28,1984.03,enviado
1,PED7001,NaN,2025-01-31,2395.24,cancelado
2,PED7002,CL1067,2024-07-13,433.46,cancelado
3,PED7003,CL1097,2025-05-02,352.01,cancelado
4,PED7004,CL1068,2024-12-20,1182.84,enviado



Columnas reales: ['id_pedido', 'id_cliente', 'fecha_pedido', 'monto', 'estado']

=== RESUMEN ===
Total procesados: 230
Curated: 190
Rejects: 40

=== ESTADOS NORMALIZADOS ===


,count
estado,
Cancelado,58
Preparacion,48
Enviado,46
Pendiente,40
Entregado,38



=== MUESTRA CURATED ===


...


Archivos generados:
- pedidos_curated.csv
- pedidos_rejects.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>